In [63]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import matplotlib.pyplot as plt
%matplotlib inline

import torch.nn.functional as F
import random

In [76]:
g = torch.Generator().manual_seed(222)

In [77]:
words = open("names.txt", "r").read().splitlines()
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [78]:
chars = sorted(list(set(''.join(words))))
mapp = {s:i+1 for i,s in enumerate(chars)}
mapp['.'] = 0
invmap = {i:s for s,i in mapp.items()}

In [79]:

def build_dataset(words):
    block_size = 5
    X,Y = [], []
    for w in words:
        #print(w)
        context = [0] * block_size
        for ch in w + '.':
            ix = mapp[ch]
            X.append(context)
            Y.append(ix)
            #print(''.join(invmap[i] for i in context), ' --> ', invmap[ix])
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

random.seed(222)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xtst, Ytst = build_dataset(words[n2:])

In [6]:
X.shape, X

(torch.Size([228146, 3]),
 tensor([[ 0,  0,  0],
         [ 0,  0,  5],
         [ 0,  5, 13],
         ...,
         [26, 26, 25],
         [26, 25, 26],
         [25, 26, 24]]))

In [7]:
Y.shape, Y

(torch.Size([228146]), tensor([ 5, 13, 13,  ..., 26, 24,  0]))

In [8]:
#each char have 2 dim embedding which is just a coordinate on graph paper for each word we assign some pos on graph paper thats it
C = torch.randn((27, 2), generator=g)
C

tensor([[-1.4609,  0.7528],
        [ 0.7871,  1.2646],
        [-0.6371, -1.1649],
        [-0.8392, -1.1444],
        [ 0.2068, -0.6295],
        [ 0.7378,  0.2925],
        [-2.2334,  0.0755],
        [ 0.4478, -1.2305],
        [ 0.6490, -0.0086],
        [-1.3489,  0.6352],
        [ 1.3309,  1.8371],
        [-0.3580,  0.1912],
        [ 0.1076,  0.8646],
        [ 0.0305,  1.1851],
        [ 0.1421, -0.0949],
        [-1.2262, -0.0694],
        [ 1.6718,  0.1663],
        [-0.4762, -1.0917],
        [ 0.5813, -0.1228],
        [ 0.2710, -1.5007],
        [ 0.6422, -0.1059],
        [-1.2223,  0.3314],
        [-0.7958,  2.1656],
        [-1.0586,  0.4177],
        [ 0.6733,  0.3785],
        [-0.2778,  0.6268],
        [-1.5342, -1.2508]])

In [9]:
#C is contianing the each characters pos on graph page and X have lots of rows with each row containing 3 characters Examples X 3 now when we do C[X]
#it replaces those each of the characters with its pos on graph so Examples X 3 X 2 its like 2 table of Example X 3 first table have x-cordinate and other have y-coordinate
C[X].shape, C[X]

(torch.Size([228146, 3, 2]),
 tensor([[[-1.4609,  0.7528],
          [-1.4609,  0.7528],
          [-1.4609,  0.7528]],
 
         [[-1.4609,  0.7528],
          [-1.4609,  0.7528],
          [ 0.7378,  0.2925]],
 
         [[-1.4609,  0.7528],
          [ 0.7378,  0.2925],
          [ 0.0305,  1.1851]],
 
         ...,
 
         [[-1.5342, -1.2508],
          [-1.5342, -1.2508],
          [-0.2778,  0.6268]],
 
         [[-1.5342, -1.2508],
          [-0.2778,  0.6268],
          [-1.5342, -1.2508]],
 
         [[-0.2778,  0.6268],
          [-1.5342, -1.2508],
          [ 0.6733,  0.3785]]]))

In [10]:
#to better understand it see the example we will take which char was at 13,2 in X table then we look what is position of that on graph page through C 
#then we will see what is at that 13,2 on C[X] which will match the position we got
X[13,2], C[X[13,2]], C[X][13,2]

(tensor(1), tensor([0.7871, 1.2646]), tensor([0.7871, 1.2646]))

In [11]:
emb = C[X]
emb = emb.view(emb.shape[0], 6)
#the weights so each example have 3 features (chars) and each feature have 2 values (coordinates) so we need 6 weights like we will flatten the emb and so will have 6 features instead
#the 100 is number of neurons in next layer
W1 = torch.randn((6, 100), generator=g)
b = torch.randn(100, generator=g)
emb.shape

torch.Size([228146, 6])

In [12]:
h = torch.tanh(emb @ W1 + b)
h.shape

torch.Size([228146, 100])

In [18]:
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)

In [19]:
logits = h@W2 + b2
counts = logits.exp()
probs = counts/counts.sum(dim=1, keepdims= True)

In [20]:
#for revision: we have some thousands of rows and each row have 27 columns, each column for a row contains a value which shows what is the prob of this char num to come next in word'
#where Y contains exact char value which came next in dataset for that row
#so probs is to be counted as sum of the row and eachh value in that row divided by this sum
probs.shape, probs[0].sum()

(torch.Size([228146, 27]), tensor(1.0000))

In [21]:
#ok now again what is the probability the neural network predicted for the correct Char that should have came means if 5th char came in 13th example then what is the prob it shows
#on the probs table at 13th example for 5th char to come cuase that should approach to 1 right...
#then calculate the loss. again since we want robs to be 1 at that pos but its surely preety low now we say we want to maximize the loss so we mini the neg of it
#also for each exampe we should multiply the probs but isntead we take log and add it and isntead of whole sum take the mean of it
idx = torch.arange(Y.shape[0])
loss = -probs[idx, Y].log().mean()
loss

tensor(15.2367)

In [22]:
#now above thing is exactly cross entropy
#see the diff throguh the below example
#x = math.exp(h), y = x+1, z = x-1, t = y/z 
#x = math.exp(h), y = (x+1)/(x-1)
#now idff between above two is in backpropagation now if i go through firts oen it willd iff wtrt next and write grad for ecah and reach x
#while second one will have to go just two back caus it have to diff only y so makes it less dense that is what is diff between coss-entropy implemntation and above implementation of loss
#also it sub the max value which came in logits to keep the counts awaya from overflowing cause more neg exponentiation is good but for pos value exp() overflows it very fast
loss = F.cross_entropy(logits, Y)
loss

tensor(15.2367)

In [105]:
C = torch.randn((27, 6), generator=g)
W1 = torch.randn((30, 200), generator=g)
b = torch.randn(200, generator=g)
W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b, W2, b2]

In [106]:
for p in parameters:
    p.requires_grad = True

In [37]:
for i in range(1000):
    #Forward pass
    emb = C[X]
    emb = emb.view(emb.shape[0], 6)
    h = torch.tanh(emb @ W1 + b)
    logits = h@W2 + b2
    loss = F.cross_entropy(logits, Y)
    if(i+1)%100 == 0:
        print(loss.item())
    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    
    for p in parameters:
        p.data += -0.1 * p.grad

loss.item()

2.7207136154174805
2.632114887237549
2.5987651348114014
2.5808005332946777
2.5691444873809814
2.560898542404175
2.5544726848602295
2.549057960510254
2.54420804977417
2.539693832397461


2.539693832397461

In [38]:
for i in range(1000):
    #Forward pass
    emb = C[X]
    emb = emb.view(emb.shape[0], 6)
    h = torch.tanh(emb @ W1 + b)
    logits = h@W2 + b2
    loss = F.cross_entropy(logits, Y)
    if(i+1)%100 == 0:
        print(loss.item())
    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    
    for p in parameters:
        p.data += -0.1 * p.grad

loss.item()

2.535419225692749
2.5313525199890137
2.527486801147461
2.523820638656616
2.520350217819214
2.51706862449646
2.5139682292938232
2.5110387802124023
2.508267402648926
2.5056426525115967


2.5056426525115967

In [116]:
#for below I ran it 3 times first two times with lr = 0.05 and then 3rd time with lr = 0.01 other thinsg were same

In [111]:
#now we are gonna use mini batch cause that is much faster
for i in range(100000):
    mb = torch.randint(0, Xtr.shape[0], (32,)) # selects random 32 rows from X 
    #Forward pass
    emb = C[Xtr[mb]]
    emb = emb.view(emb.shape[0],30)
    h = torch.tanh(emb @ W1 + b)
    logits = h@W2 + b2
    loss = F.cross_entropy(logits, Ytr[mb])
    if(i+1)%10000 == 0:
        print(loss.item())
    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    
    for p in parameters:
        p.data += -0.01 * p.grad

2.428802013397217
2.22528338432312
1.965462565422058
1.8997548818588257
2.354187488555908
2.053140640258789
1.9061135053634644
2.176442861557007
2.178837299346924
1.890849232673645


In [112]:
emb = C[Xtr]
emb = emb.view(emb.shape[0], 30)
h = torch.tanh(emb @ W1 + b)
logits = h@W2 + b2
loss = F.cross_entropy(logits, Ytr)
loss.item()

2.162553548812866

In [113]:
emb = C[Xdev]
emb = emb.view(emb.shape[0], 30)
h = torch.tanh(emb @ W1 + b)
logits = h@W2 + b2
loss = F.cross_entropy(logits, Ydev)
loss.item()

2.185767889022827

In [114]:
emb = C[Xtst]
emb = emb.view(emb.shape[0], 30)
h = torch.tanh(emb @ W1 + b)
logits = h@W2 + b2
loss = F.cross_entropy(logits, Ytst)
loss.item()

2.1761162281036377

In [115]:
# Sampling from the neural network
g = torch.Generator().manual_seed(222)
block_size = 5
for _ in range(20):
    out = []
    context = [0] * block_size 
    
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, 30) @ W1 + b)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        
        context = context[1:] + [ix]
        out.append(ix)
        
        if ix == 0:
            break

    print(''.join(invmap[i] for i in out))

nexim.
zkalf.
roidama.
crishton.
ranavderly.
jhlip.
collis.
craquemio.
jaylean.
tcarlich.
aliar.
uzan.
evion.
kenia.
marlinna.
artelishan.
avirca.
homarie.
nakar.
kabeezin.
